# 4、工具的应用案例
## 4.1案例1：使用args_schema

In [1]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from rich import print as rprint
from scripts.regsetup import description

# 将env文件中的变量加载为环境变量
#override=True：表示.env优先
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
model = ChatDeepSeek(
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    model_name="deepseek-v4-flash"
)

In [5]:

from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from pydantic import BaseModel,Field
class WeatherSchema(BaseModel):
    city:str=Field(default="北京",description="具体的城市名称")
    if_forecast:bool=Field(default=False,description="是否包含明日天气")
@tool("get_weather_and_forecast",description="查询当日的天气，可以包含明天的天气预报",
      args_schema=WeatherSchema)
def get_weather(city:str,if_forecast:bool):
    res=f"{city}今天天气不错"
    if if_forecast:
        res+="\n明天下雨"
    return res
print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weather_and_forecast', 'description': '查询当日的天气，可以包含明天的天气预报', 'parameters': {'properties': {'city': {'default': '北京', 'description': '具体的城市名称', 'type': 'string'}, 'if_forecast': {'default': False, 'description': '是否包含明日天气', 'type': 'boolean'}}, 'type': 'object'}}}


In [6]:
from langchain_core.messages import HumanMessage

# 1.将工具绑定到模型上
model_with_wools=model.bind_tools([get_weather])
#2.维护一个消息列表
messages=[HumanMessage("今天杭州的天气怎么样？明天呢？")]
#3.调用模型，得到响应：AIMessage
response=model_with_wools.invoke(messages)
messages.append(response)
#4.获取响应中的tool_calls字段信息
tool_calls=response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"]=="get_weather_and_forecast":
        #5.调用工具，调用完返回TooMessage的实例
        tool_message=get_weather.invoke(tool_call)
        messages.append(tool_message)
#6.调用模型，得到AIMessage
final_response=model.invoke(messages)
#7.添加到消息列表中
messages.append(final_response)
#8.便利消息列表
for msg in messages:
    msg.pretty_print()

================================ Human Message =================================

今天杭州的天气怎么样？明天呢？
================================== Ai Message ==================================

我来帮您查询杭州今天和明天的天气情况。
Tool Calls:
  get_weather_and_forecast (call_00_9oEgRwSHl5EXhklLFHy98195)
 Call ID: call_00_9oEgRwSHl5EXhklLFHy98195
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

杭州今天天气不错
明天下雨
================================== Ai Message ==================================

查询到杭州的天气情况如下：

- **今天**：天气不错
- **明天**：有雨

温馨提示：明天出门记得带伞，注意雨天出行安全。如果您需要更具体的风力、气温等数据，也可以告诉我，我再帮您查询～


## 2、多工具调用

In [8]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1.定义工具
# 定义股票查询工具
@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
    """获取指定公司的股票价格信息

    Args:
        company: 公司名称（如：苹果公司, 微软公司, 谷歌公司）
        timeframe: 时间范围（today-今日, week-本周, month-本月）
    """
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
        "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
    }
    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe}价格: {price}美元"
    else:
        return f"未找到股票代码 {company} 的数据"


# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """搜索指定公司的财经新闻

    Args:
        company: 公司名称

    Returns:
        公司的财经新闻，每个新闻占一行
    """
    # 模拟新闻数据
    mock_news = {
        "苹果公司": [
            "苹果发布新款iPhone，股价上涨3%",
            "苹果与欧盟达成反垄断和解协议",
            "苹果将在印度扩大生产规模"
        ],
        "微软公司": [
            "微软Azure云业务季度增长超预期",
            "微软完成对Nuance的收购",
            "微软推出新一代AI助手Copilot"
        ],
        "谷歌公司": [
            "谷歌发布新AI模型，性能提升20%",
            "谷歌与OpenAI合作，开发新的AI助手",
            "谷歌在欧洲展开AI研究项目"
        ]
    }
    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)


# rprint(convert_to_openai_tool(search_news))
# 2.初始化模型并绑定工具
tools = [get_stock_price, search_news]
model_with_tools = model.bind_tools(tools)
message_list = []
human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻？")
# human_message = HumanMessage(content="比较一下微软和苹果的股价")
# human_message = HumanMessage(content="腾讯最近有什么重大新闻？")
# human_message = HumanMessage(content="海水为什么是咸的？")
message_list.append(human_message)

# 3.工具调用
while True:
    response = model_with_tools.invoke(message_list)
    message_list.append(response)
    # 如果模型不需要调用工具，直接退出循环
    if not response.tool_calls:
        print("没有工具调用，直接返回答案")
        break

    # 如果有调用工具，处理工具调用响应
    # 4.开发者根据模型的响应，调用工具并获取结果
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_stock_price":
            stock_result = get_stock_price.invoke(tool_call)
            print("stock_result", stock_result)
            message_list.append(stock_result)
        if tool_call["name"] == "search_news":
            news_result = search_news.invoke(tool_call)
            print("news_result", news_result)
            message_list.append(news_result)

# print("response", response)
# print(response.content)
for msg in message_list:
    msg.pretty_print()

stock_result content='苹果公司 today价格: 185.2美元' name='get_stock_price' tool_call_id='call_00_HnZ9mPEV814catSHCHoK6952'
news_result content='苹果发布新款iPhone，股价上涨3%\n苹果与欧盟达成反垄断和解协议\n苹果将在印度扩大生产规模' name='search_news' tool_call_id='call_01_rzT95ZnY62Zv8yyGDeB15469'
没有工具调用，直接返回答案
================================ Human Message =================================

苹果公司今天的股价是多少？最近有什么新闻？
================================== Ai Message ==================================

我来为您查询苹果公司的股价和最新新闻。
Tool Calls:
  get_stock_price (call_00_HnZ9mPEV814catSHCHoK6952)
 Call ID: call_00_HnZ9mPEV814catSHCHoK6952
  Args:
    company: 苹果公司
    timeframe: today
  search_news (call_01_rzT95ZnY62Zv8yyGDeB15469)
 Call ID: call_01_rzT95ZnY62Zv8yyGDeB15469
  Args:
    company: 苹果公司
================================= Tool Message =================================
Name: get_stock_price

苹果公司 today价格: 185.2美元
================================= Tool Message =================================
Name: search_news

苹果发布新款iPhone，股价上涨3%
苹果与欧盟达

## 3、多工具调用

In [10]:
from langchain.tools import tool
from langchain.messages import HumanMessage


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当日天气

    Args:
        city: 城市名称
    """
    return f'{city}当天晴朗'


@tool(parse_docstring=True)
def get_news() -> str:
    """
    获取当日新闻
    """
    return "近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。"


model_with_tools = model.bind_tools([get_weather, get_news])
messages = [
    HumanMessage("今天杭州天气如何？今天新闻是什么？别瞎编")
]
response = model_with_tools.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================

我来为您查询杭州今天的天气和今日新闻。
Tool Calls:
  get_weather (call_00_S62QkP0WTDvQUrPk460W2106)
 Call ID: call_00_S62QkP0WTDvQUrPk460W2106
  Args:
    city: 杭州
  get_news (call_01_cQJnspVjzvfSMEYvDLaz2886)
 Call ID: call_01_cQJnspVjzvfSMEYvDLaz2886
  Args:


In [11]:
messages.append(response)
# 输出
for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_msg = get_weather.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    elif tool_call["name"] == "get_news":
        tool_msg = get_news.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    else:
        raise Exception("不存在的工具")

final_response = model.invoke(messages)
messages.append(final_response)
for msg in messages:
    msg.pretty_print()

content='杭州当天晴朗' name='get_weather' tool_call_id='call_00_S62QkP0WTDvQUrPk460W2106'
content='近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。' name='get_news' tool_call_id='call_01_cQJnspVjzvfSMEYvDLaz2886'
================================ Human Message =================================

今天杭州天气如何？今天新闻是什么？别瞎编
================================== Ai Message ==================================

我来为您查询杭州今天的天气和今日新闻。
Tool Calls:
  get_weather (call_00_S62QkP0WTDvQUrPk460W2106)
 Call ID: call_00_S62QkP0WTDvQUrPk460W2106
  Args:
    city: 杭州
  get_news (call_01_cQJnspVjzvfSMEYvDLaz2886)
 Call ID: call_01_cQJnspVjzvfSMEYvDLaz2886
  Args:
================================= Tool Message =================================
Name: get_weather

杭州当天晴朗
================================= Tool Message =================================
Name: get_news

近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。
================================== Ai Message ==================================

好的，这是